# Spotify Data Analysis — Part 2: Data Cleaning

This notebook covers the **inspection and cleaning** of the raw Spotify data:

1. Data loading and shape inspection
2. Missing value analysis
3. Release year handling
4. Duration normalization
5. Language detection (Hebrew vs English)
6. Lyrics text cleaning
7. Final data quality summary

In [ ]:
import sqlite3
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import io
from IPython.display import SVG, display as _ipy_display

DB_PATH = "spotify_intel.db"

def q(sql):
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn)

# SVG rendering — fixes Hebrew BiDi text
matplotlib.rcParams['svg.fonttype'] = 'none'

def _svg_show(*args, **kwargs):
    buf = io.BytesIO()
    plt.savefig(buf, format='svg', bbox_inches='tight')
    buf.seek(0)
    _ipy_display(SVG(buf.read()))
    plt.close()

plt.show = _svg_show

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'font.size': 11,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Segoe UI', 'Tahoma', 'Arial', 'DejaVu Sans'],
})

C1 = '#6c5ce7'
C2 = '#dfe6e9'
print("Setup complete.")

---
## 1. Load & Inspect the Tracks Table

In [ ]:
# Load full tracks table
tracks = q("SELECT * FROM tracks")
print(f"Shape: {tracks.shape[0]} rows × {tracks.shape[1]} columns")
print(f"\nColumn types:")
print(tracks.dtypes)
tracks.head()

---
## 2. Missing Value Analysis

In [ ]:
# Count nulls per column
null_counts = tracks.isnull().sum().reset_index()
null_counts.columns = ['column', 'null_count']
null_counts['null_pct'] = (null_counts['null_count'] / len(tracks) * 100).round(1)
null_counts = null_counts[null_counts['null_count'] > 0].sort_values('null_count', ascending=False)
display(null_counts)

# Visualise
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(null_counts['column'], null_counts['null_pct'], color=C1)
ax.set_xlabel('% missing')
ax.set_title('Missing Values per Column', fontsize=13, fontweight='bold')
ax.set_xlim(0, 105)
for i, pct in enumerate(null_counts['null_pct']):
    ax.text(pct + 1, i, f'{pct}%', va='center', fontsize=10)
plt.tight_layout()
plt.show()

### Observations

- **`popularity`**: All NULL — Spotify does not return this field for Israeli/regional tracks (a known API limitation).
- **`preview_url`**: ~99% NULL — preview clips are region-restricted and rarely returned.
- **`audio_features`**: Not collected in this pipeline run.
- **`release_year`**: A small number of tracks have no year — these are kept with NULL.

---
## 3. Release Year Analysis

In [ ]:
# Distribution of release years
yr = q("""
SELECT
    release_year,
    COUNT(*) AS tracks
FROM tracks
WHERE release_year IS NOT NULL
GROUP BY release_year
ORDER BY release_year
""")

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(yr['release_year'], yr['tracks'], color=C1, width=0.8)
ax.set_xlabel('Release Year')
ax.set_ylabel('Tracks')
ax.set_title('Tracks by Release Year', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Earliest: {int(yr['release_year'].min())}  |  Latest: {int(yr['release_year'].max())}  |  Tracks without year: {tracks['release_year'].isnull().sum()}")

---
## 4. Duration Normalization

Raw durations from the Spotify API are in **milliseconds**. We convert to mm:ss for display and to seconds for analysis.

In [ ]:
dur = q("""
SELECT
    ROUND(MIN(duration_ms) / 1000.0, 0)  AS min_sec,
    ROUND(AVG(duration_ms) / 1000.0, 0)  AS avg_sec,
    ROUND(MAX(duration_ms) / 1000.0, 0)  AS max_sec,
    COUNT(*) - COUNT(duration_ms)         AS null_durations
FROM tracks
""")
display(dur)

# Duration distribution
dur_all = q("SELECT duration_ms / 1000.0 AS duration_sec FROM tracks WHERE duration_ms IS NOT NULL")
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(dur_all['duration_sec'], bins=40, color=C1, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Duration (seconds)')
ax.set_ylabel('Tracks')
ax.set_title('Song Duration Distribution', fontsize=13, fontweight='bold')
ax.axvline(dur['avg_sec'][0], color='#e17055', linestyle='--', linewidth=2, label=f'Avg: {int(dur["avg_sec"][0])}s')
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Language Detection — Hebrew vs English

Artist names and song titles contain both Hebrew and Latin characters. A Unicode range check classifies each track:

In [ ]:
# Detect Hebrew vs English tracks
tracks_lang = q("SELECT name, artist FROM tracks")

def has_hebrew(text):
    if not isinstance(text, str): return False
    return any('\u05d0' <= ch <= '\u05ea' for ch in text)

tracks_lang['hebrew_title']  = tracks_lang['name'].apply(has_hebrew)
tracks_lang['hebrew_artist'] = tracks_lang['artist'].apply(has_hebrew)
tracks_lang['is_hebrew']     = tracks_lang['hebrew_title'] | tracks_lang['hebrew_artist']

summary = tracks_lang['is_hebrew'].value_counts().rename({True: 'Hebrew', False: 'English / Other'})
display(summary.to_frame('tracks'))

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(summary, labels=summary.index, colors=[C1, C2],
       autopct='%1.1f%%', startangle=90, wedgeprops=dict(width=0.5))
ax.set_title('Hebrew vs Non-Hebrew Tracks', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Sample Hebrew titles and artists
hebrew_sample = tracks_lang[tracks_lang['is_hebrew']].head(10)[['name','artist']]
print("Sample Hebrew-language tracks:")
display(hebrew_sample)

---
## 6. Lyrics Cleaning

Raw lyrics scraped from Genius and Shironet contain HTML artifacts, section markers (`[Chorus]`, `[Verse]`), and extra whitespace. The cleaning pipeline:

In [ ]:
# Raw vs cleaned lyrics sample
lyrics_sample = q("""
SELECT
    t.name,
    t.artist,
    length(tl.lyrics_raw)     AS raw_chars,
    length(tl.lyrics_cleaned) AS cleaned_chars,
    ROUND(
        (1.0 - CAST(length(tl.lyrics_cleaned) AS FLOAT) / length(tl.lyrics_raw)) * 100,
        1
    ) AS pct_reduction
FROM track_lyrics tl
JOIN tracks t ON tl.track_id = t.id
WHERE tl.lyrics_raw IS NOT NULL AND tl.lyrics_cleaned IS NOT NULL
ORDER BY pct_reduction DESC
LIMIT 10
""")
display(lyrics_sample)

In [ ]:
# Cleaning steps (reference code)
#
# import re
#
# def clean_lyrics(raw: str) -> str:
#     text = re.sub(r'<[^>]+>', ' ', raw)          # strip HTML tags
#     text = re.sub(r'\[.*?\]', '', text)            # remove section markers [Chorus], etc.
#     text = re.sub(r'\n{3,}', '\n\n', text)         # collapse excess blank lines
#     text = re.sub(r' {2,}', ' ', text)             # collapse spaces
#     return text.strip()

print("Cleaning steps: HTML removal → section markers → whitespace normalization")

---
## 7. Final Data Quality Summary

In [ ]:
summary_q = q("""
SELECT
    (SELECT COUNT(*) FROM tracks)                                        AS total_tracks,
    (SELECT COUNT(*) FROM tracks WHERE release_year IS NOT NULL)         AS tracks_with_year,
    (SELECT COUNT(*) FROM tracks WHERE duration_ms IS NOT NULL)          AS tracks_with_duration,
    (SELECT COUNT(*) FROM track_lyrics WHERE lyrics_raw IS NOT NULL)     AS tracks_with_lyrics,
    (SELECT COUNT(*) FROM track_lyrics WHERE lyrics_cleaned IS NOT NULL) AS tracks_with_clean_lyrics,
    (SELECT COUNT(*) FROM listening_snapshot WHERE term = 'liked')       AS liked_songs
""")

for col in summary_q.columns:
    val = int(summary_q[col][0])
    pct = round(val / 725 * 100, 1)
    print(f"{col:<35} {val:>4}  ({pct}% of all tracks)")